# BCI2000 / Neuro Omega Neural Signal Checker (corrected)

This notebook checks for continuous neural-source signals in a BCI2000 `.dat` recording, including Neuro Omega `LFP 01` and `Macro LFP 01`.

**Important fix:** Sample records contain **both signal bytes and BCI2000 state-vector bytes**. In the first version of this notebook, the parser did not recognize the `StatevectorLen` spelling in some headers and mistakenly treated state bytes as neural data. This version recognizes `StatevectorLen` and `StateVectorLength`, validates sample record size, and supports sampling rates such as `1.375k` = 1375 Hz.

Features: source metadata; sampled-window channel quality; waveforms; spectral analysis; full-recording LFP plots with peak-preserving downsampling; optional independent validation using `BCI2000Tools` (if installed). A varying signal is **not** proof of neural origin or signal quality.

**Only the raw BCI2000 file is required.** The `BCI2000Tools` verification is optional.

## 1. Set your file path and preferences

Change **`DAT_PATH`** to your recording. Other settings can stay as they are for a first check.

In [ ]:
from pathlib import Path

# CHANGE THIS ONE LINE to the location of your BCI2000 .dat file.
DAT_PATH = Path(r"C:\Users\gabri\Desktop\joystick_test_env\python_da\BIDS_env\data_dir\neuroomega\NameS001R04.dat")

# Check this many seconds at the beginning, middle, and end of the recording.
WINDOW_SECONDS = 3.0

# Select channels using 0-based indices (0 means the first channel).
CHANNELS_TO_PLOT = [0, 1, 2, 3]
PLOT_START_SECONDS = 0.0
PLOT_DURATION_SECONDS = 2.0

# Maximum frequency shown in the PSD plot (Nyquist limit also applies).
PSD_MAX_HZ = 200.0

# Only write a CSV if you explicitly enable this.
EXPORT_CSV = False

## 2. Read the header and open the signals

BCI2000 stores signal samples from all channels followed by a *state vector* for each sample. This reader uses that layout and a memory map to access arbitrary time windows without loading the entire file.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from urllib.parse import unquote
from IPython.display import display


def extract_parameter_line(header_text, name):
    """Return the text after 'ParameterName=' from the header, if found."""
    match = re.search(r"(?m)^.*?\b" + re.escape(name) + r"\s*=\s*([^\r\n]*)", header_text)
    return match.group(1).strip() if match else None


def read_number_parameter(header_text, name):
    """Parse a BCI2000 numeric parameter, including SI suffixes (1.375k = 1375)."""
    line = extract_parameter_line(header_text, name)
    if line is None:
        return None
    # Numeric value plus optional multiplier, with or without Hz.
    match = re.match(r"^\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*([pnumkMGTµ]?)", line)
    if not match:
        return None
    scale = {'': 1, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'µ': 1e-6,
             'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9, 'T': 1e12}
    return float(match.group(1)) * scale[match.group(2)]


def read_channel_names(header_text, n_channels):
    """BCI2000's list syntax normally begins with the number of entries."""
    line = extract_parameter_line(header_text, "ChannelNames")
    if line:
        body = line.split("//", 1)[0].strip()
        tokens = body.split()
        if tokens and tokens[0].isdigit():
            count = int(tokens[0])
            if count == n_channels and len(tokens) >= count + 1:
                return [unquote(token) for token in tokens[1:count + 1]]
    return [f"Ch{i + 1:03d}" for i in range(n_channels)]


def read_state_names(header_text):
    pattern = r"\[\s*State\s+Vector\s+Definition\s*\](.*?)(?:\[\s*Parameter\s+Definition\s*\]|\Z)"
    match = re.search(pattern, header_text, re.I | re.S)
    if not match:
        return []
    names = []
    for line in match.group(1).splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[1].isdigit():
            names.append(parts[0])
    return names


def open_bci2000_dat(path):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}\nEdit DAT_PATH in section 1.")

    with path.open("rb") as handle:
        first_line = handle.readline(4096).decode("ascii", errors="replace")
        if not ("HeaderLen" in first_line and "SourceCh" in first_line):
            raise ValueError("This does not look like a BCI2000 .dat file (header fields missing).")
        def header_field(*names):
            for name in names:
                match = re.search(r"\b" + re.escape(name) + r"\s*=\s*(\S+)", first_line, re.IGNORECASE)
                if match:
                    return match.group(1)
            return None
        header_len = int(header_field("HeaderLen") or 0)
        channel_count = int(header_field("SourceCh") or 0)
        # BCI2000 files use either StatevectorLen or StateVectorLength.
        # Missing state length must be a fatal error, NEVER silently assume zero.
        state_len_text = header_field("StatevectorLen", "StateVectorLength", "StateVectorLen")
        if state_len_text is None:
            raise ValueError("Cannot find StatevectorLen / StateVectorLength in the first header line.")
        state_len = int(state_len_text)
        # BCI2000 1.0 files without a DataFormat field are int16 by specification.
        raw_format = (header_field("DataFormat") or "int16").lower()
        if not (0 < header_len < 100_000_000) or channel_count <= 0 or state_len <= 0:
            raise ValueError("Invalid BCI2000 header dimensions (SourceCh and state-vector bytes must be > 0).")
        if raw_format not in {"int16", "int32", "float32"}:
            raise ValueError(f"Unsupported BCI2000 DataFormat: {raw_format}")
        handle.seek(0)
        header = handle.read(header_len).decode("latin-1", errors="replace")

    sampling_rate = read_number_parameter(header, "SamplingRate")
    if sampling_rate is None or sampling_rate <= 0:
        raise ValueError("Could not parse SamplingRate from the BCI2000 header.")

    raw_dtype = {"int16": "<i2", "int32": "<i4", "float32": "<f4"}[raw_format]
    record_dtype = np.dtype([
        ("signals", raw_dtype, (channel_count,)),
        ("states", "u1", (state_len,)),
    ], align=False)
    bytes_after_header = path.stat().st_size - header_len
    if bytes_after_header < 0 or record_dtype.itemsize == 0:
        raise ValueError("The file does not contain valid sample records.")
    n_samples, leftover = divmod(bytes_after_header, record_dtype.itemsize)
    if n_samples == 0:
        raise ValueError("No complete signal samples found in this file.")
    if leftover:
        print(f"WARNING: {leftover} trailing bytes do not constitute a complete {record_dtype.itemsize}-byte sample record.")
    recordings = np.memmap(path, dtype=record_dtype, mode="r", offset=header_len, shape=(n_samples,))
    return {
        "path": path, "header": header, "records": recordings,
        "n_channels": channel_count, "n_samples": int(n_samples),
        "sample_rate": float(sampling_rate), "format": raw_format,
        "state_vector_bytes": state_len, "bytes_per_record": record_dtype.itemsize,
        "header_bytes": header_len,
        "leftover_bytes": leftover, "file_bytes": path.stat().st_size,
        "names": read_channel_names(header, channel_count),
        "states": read_state_names(header),
        "sample_block_size": read_number_parameter(header, "SampleBlockSize"),
    }


file_info = open_bci2000_dat(DAT_PATH)
records = file_info["records"]
fs = file_info["sample_rate"]
n_channels = file_info["n_channels"]
n_samples = file_info["n_samples"]
channel_names = file_info["names"]

print("File:", file_info["path"])
print("Size: {:.1f} MiB".format(file_info["file_bytes"] / 1024**2))
print("Continuous signal channels:", n_channels)
print("Sampling rate: {:g} Hz".format(fs))
print("Samples per channel:", f"{n_samples:,}")
print("Duration: {:.2f} seconds ({:.2f} minutes)".format(n_samples / fs, n_samples / fs / 60))
print("Stored sample format:", file_info["format"])
print("State vector bytes per sample:", file_info["state_vector_bytes"])
print("Bytes per complete record:", file_info["bytes_per_record"])
print("SampleBlockSize:", file_info["sample_block_size"])
print("Recorded states:", len(file_info["states"]))
if file_info["leftover_bytes"]:
    print("WARNING:", file_info["leftover_bytes"], "extra bytes after the last complete sample; possible truncated file.")

print("\nFirst channel names:", channel_names[:min(12, n_channels)])
print("\nState names (these are NOT neural channels):")
print(", ".join(file_info["states"]) if file_info["states"] else "No state names found in header")

## 3. Verify the data reader (recommended)

The previous version incorrectly treated BCI2000 state-vector bytes as signal data. This cell independently compares the corrected raw memory-map reader with the existing `BCI2000Tools` library **for a small portion of the file**. The reported sample count should match. Since `BCI2000Tools` may apply channel gains/offsets, agreement within floating-point tolerance is also checked; the header currently reports unity gains and zero offsets on your Neuro Omega channels.

In [ ]:
print("Corrected sample count:", f"{n_samples:,}")
print("Corrected duration (min):", round(n_samples / fs / 60, 3))
print("Stored state bytes per sample:", file_info["state_vector_bytes"])

try:
    from BCI2000Tools.FileReader import bcistream
except ImportError:
    print("BCI2000Tools not installed; independent validation skipped.")
else:
    ref_stream = bcistream(str(DAT_PATH))
    print("BCI2000Tools sample count:", f"{ref_stream.samples():,}")
    print("BCI2000Tools sampling rate:", ref_stream.samplingfreq_hz)
    print("Same total sample count:", int(ref_stream.samples()) == n_samples)
    test_n = min(1000, n_samples)
    ref_sig, _ = ref_stream.decode(nsamp=test_n)
    ref_sig = np.asarray(ref_sig)
    raw_sig = np.asarray(records[:test_n]["signals"], dtype=float).T
    print("Reference shape:", ref_sig.shape, "; Raw shape:", raw_sig.shape)
    if ref_sig.shape == raw_sig.shape:
        for name in ["LFP 01", "Macro LFP 01"]:
            if name in channel_names:
                i = channel_names.index(name)
                a, b = np.asarray(ref_sig[i], dtype=float), raw_sig[i]
                print(f"{name}: exact={np.array_equal(a, b)}, close={np.allclose(a, b)}, max_abs_diff={np.max(np.abs(a-b)):g}")
    else:
        print("CAUTION: Reader shapes differ. Do not conclude samples agree until orientation/indexing is checked.")


## 3. Check the start, middle, and end

These measurements come from a few **sampled windows**, not every sample of the file. Data units are **raw stored values**, before applying `SourceChGain` and `SourceChOffset`. A constant signal is suspicious, but a changing signal does **not** prove that it originates from neurons.

In [ ]:
if n_channels == 0:
    raise ValueError("This recording declares zero continuous signal channels. No neural channel waveforms can be examined.")

window_samples = max(1, min(n_samples, int(round(WINDOW_SECONDS * fs))))
starts = sorted(set([0, max(0, (n_samples - window_samples) // 2), max(0, n_samples - window_samples)]))
windows = []
for start in starts:
    stop = start + window_samples
    # This is small even if the entire .dat is many GB.
    arr = np.asarray(records[start:stop]["signals"], dtype=np.float64).T
    windows.append(arr)
    print(f"Read {arr.shape[1]:,} samples/ch from {start / fs:.2f} to {stop / fs:.2f} s")

sampled = np.concatenate(windows, axis=1)
summary_rows = []
int_format = file_info["format"] in {"int16", "int32"}
if int_format:
    limits = np.iinfo(np.int16 if file_info["format"] == "int16" else np.int32)

for i in range(n_channels):
    x = sampled[i]
    good = x[np.isfinite(x)]
    finite_fraction = good.size / max(1, x.size)
    if good.size:
        mean, std = float(np.mean(good)), float(np.std(good))
        minval, maxval = float(good.min()), float(good.max())
        unique = len(np.unique(good))
        zero_fraction = float(np.mean(good == 0))
        # Only count adjacent samples within the SAME window.
        steps = [np.diff(w[i]) for w in windows if w.shape[1] > 1]
        equal_fraction = (float(np.mean(np.concatenate(steps) == 0)) if steps else np.nan)
        saturated_fraction = float(np.mean((good == limits.min) | (good == limits.max))) if int_format else np.nan
        if unique == 1:
            assessment = "CONSTANT"
        elif unique <= 3:
            assessment = "LOW_VARIATION"
        else:
            assessment = "VARIABLE"
    else:
        mean = std = minval = maxval = zero_fraction = equal_fraction = saturated_fraction = np.nan
        unique = 0
        assessment = "NO_FINITE_DATA"
    summary_rows.append({
        "index_0_based": i,
        "channel": channel_names[i],
        "status": assessment,
        "n_unique": unique,
        "std_raw": std,
        "min_raw": minval,
        "max_raw": maxval,
        "mean_raw": mean,
        "zero_fraction": zero_fraction,
        "repeated_step_fraction": equal_fraction,
        "rail_fraction": saturated_fraction,
        "finite_fraction": finite_fraction,
    })

quality_df = pd.DataFrame(summary_rows)
print("\nChannel status counts (sampled windows only):")
print(quality_df["status"].value_counts().to_string())
print("\nPer-channel summary:")
display(quality_df)

if EXPORT_CSV:
    save_path = file_info["path"].with_name(file_info["path"].stem + "_neural_channel_check.csv")
    quality_df.to_csv(save_path, index=False)
    print("Saved:", save_path)

## 4. Overall channel variability

Standard deviation summarizes how much each stored channel changes in the sampled windows. It is **not** a measure of neural recording quality on its own. Channel numbers on the horizontal axis are **1-based** for readability.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(np.arange(1, n_channels + 1), quality_df["std_raw"], ".", markersize=4)
ax.set_xlabel("Channel number (1-based)")
ax.set_ylabel("Standard deviation (raw stored units)")
ax.set_title("Variability by continuous signal channel")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Most variable channels:")
display(quality_df.sort_values("std_raw", ascending=False).head(min(10, n_channels)))
print("\nConstant and low-variation channels:")
display(quality_df[quality_df["status"].isin(["CONSTANT", "LOW_VARIATION", "NO_FINITE_DATA"])])

## 5. Plot the raw signal waveforms

Change `CHANNELS_TO_PLOT`, `PLOT_START_SECONDS`, and `PLOT_DURATION_SECONDS` at the top, then rerun this cell. Each channel has its **own** y-axis scale, so small signals stay visible.

In [ ]:
selected = [i for i in CHANNELS_TO_PLOT if isinstance(i, (int, np.integer)) and 0 <= i < n_channels]
selected = list(dict.fromkeys(selected))
if not selected:
    print("No valid channel indices selected. Try CHANNELS_TO_PLOT = [0].")
else:
    first_sample = max(0, min(int(PLOT_START_SECONDS * fs), n_samples - 1))
    last_sample = min(n_samples, first_sample + max(1, int(PLOT_DURATION_SECONDS * fs)))
    x_time = np.arange(first_sample, last_sample) / fs
    fig, axes = plt.subplots(len(selected), 1, figsize=(13, 2.3 * len(selected)), sharex=True, squeeze=False)
    for ax, i in zip(axes.flat, selected):
        values = np.asarray(records[first_sample:last_sample]["signals"][:, i], dtype=float)
        ax.plot(x_time, values, lw=0.65)
        ax.set_ylabel(f"{channel_names[i]}\n(raw)")
        ax.grid(alpha=0.2)
    axes[0, 0].set_title("Raw stored continuous signals")
    axes[-1, 0].set_xlabel("Recording time (seconds)")
    plt.tight_layout()
    plt.show()

## 6. Frequency content of the selected channels

Welch power spectral density (PSD) helps distinguish flat channels, narrowband interference, and broader physiological-like activity. **A 60 Hz peak alone is not evidence of neural activity.** Frequencies above `sampling_rate / 2` cannot be reconstructed from the recording.

In [ ]:
if not selected:
    print("Select at least one valid channel index in section 1.")
else:
    # The same sampled windows used in the quality table; no large file read.
    freq_cap = min(PSD_MAX_HZ, fs / 2)
    fig, ax = plt.subplots(figsize=(12, 5))
    did_plot = False
    for i in selected:
        vec = sampled[i].astype(float)
        if not np.isfinite(vec).all() or vec.size < 8 or np.std(vec) == 0:
            print(f"Skipping PSD for channel {i} ({channel_names[i]}): insufficient nonconstant finite data")
            continue
        # Welch on each separate recording window to avoid false boundaries at window joins.
        individual_psds = []
        for win in windows:
            v = win[i]
            if len(v) < 8 or not np.isfinite(v).all():
                continue
            freq, psd = welch(v, fs=fs, nperseg=min(2048, len(v)), detrend="constant")
            individual_psds.append(psd)
        if not individual_psds:
            continue
        mean_psd = np.mean(individual_psds, axis=0)
        mask = (freq > 0) & (freq <= freq_cap)
        ax.semilogy(freq[mask], np.maximum(mean_psd[mask], np.finfo(float).tiny), label=channel_names[i])
        did_plot = True
    if did_plot:
        ax.set_title("Welch PSD (sampled windows)")
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("PSD (raw stored units²/Hz)")
        ax.grid(alpha=0.25)
        ax.legend(loc="best")
        plt.tight_layout()
        plt.show()
    else:
        plt.close(fig)
        print("No valid PSD curves available.")
    print(f"Sampling rate = {fs:g} Hz; Nyquist frequency = {fs / 2:g} Hz")

## 7. Interpreting the result

This test answers the initial **data-presence** question. It does **not** establish signal provenance or physiological validity.

- **Constant/all-zero channels:** Might be unconnected electrodes, placeholder channels, or test/source configuration issues.
- **Variable channels:** Time-varying values really are stored in the `.dat`. They could be neural data, electrical noise, another sensor, or synthetic output.
- **High `rail_fraction`:** Possible numeric saturation when using integer data; inspect carefully.
- **High `repeated_step_fraction`:** May indicate long flat intervals, but integer quantization can also produce repeats.
- **PSD:** LFP-like frequencies can be examined at 2 kHz, but conventional high-frequency microelectrode spikes generally need a higher sample rate. The maximum observable frequency is always less than or equal to Nyquist.

To **confirm that channels contain neural recordings**, compare these waveforms and header channel names against your known amplifier/source setup and, ideally, recordings made with electrodes connected and disconnected or with known neural activity.

In [ ]:
n_variable = int((quality_df["status"] == "VARIABLE").sum())
n_low = int((quality_df["status"] == "LOW_VARIATION").sum())
n_constant = int((quality_df["status"] == "CONSTANT").sum())
n_nonfinite = int((quality_df["status"] == "NO_FINITE_DATA").sum())

print("=== DATA-PRESENCE ASSESSMENT ===")
print(f"Declared continuous channels: {n_channels}")
print(f"Variable sampled channels:    {n_variable}")
print(f"Low-variation channels:       {n_low}")
print(f"Constant sampled channels:    {n_constant}")
print(f"No-finite-data channels:      {n_nonfinite}")

if n_variable > 0:
    print("\nRESULT: YES, changing continuous signal values are stored in this .dat file.")
    print("Neural origin: NOT VERIFIED. Check source module/electrode mapping and waveforms.")
elif n_low > 0:
    print("\nRESULT: Some signal channels change, but show very few distinct values.")
    print("Neural origin: NOT VERIFIED. Investigate raw traces and acquisition configuration.")
else:
    print("\nRESULT: No clearly varying continuous signals were seen in the sampled windows.")
    print("This does not rule out activity elsewhere in the file. Increase WINDOW_SECONDS or choose different windows.")
print("\nInterpretation is limited to the sampled windows and raw stored values.")

## 8. Full-recording activity: `LFP 01` and `Macro LFP 01`

Plots minimum and maximum raw values in each time bin, preserving peaks while avoiding plotting millions of points. This is **an amplitude envelope**, not a continuous sample-by-sample trace. It reads the corrected signal stream including the 101-byte state vector in each sample record. Scale remains in stored raw units; source gains/offsets from the header are not applied.

In [ ]:
FULL_RECORDING_CHANNELS = ["LFP 01", "Macro LFP 01"]
MAX_PLOT_BINS = 8000

missing = [name for name in FULL_RECORDING_CHANNELS if name not in channel_names]
if missing:
    raise ValueError(f"Channels not found: {missing}. Available: {channel_names}")

ch_ids = [channel_names.index(name) for name in FULL_RECORDING_CHANNELS]
bin_size = max(1, int(np.ceil(n_samples / MAX_PLOT_BINS)))
n_bins = int(np.ceil(n_samples / bin_size))
mins = np.empty((len(ch_ids), n_bins), dtype=float)
maxs = np.empty_like(mins)

for j in range(n_bins):
    start = j * bin_size
    stop = min(start + bin_size, n_samples)
    segment = np.asarray(records[start:stop]["signals"][:, ch_ids], dtype=float)
    mins[:, j] = np.min(segment, axis=0)
    maxs[:, j] = np.max(segment, axis=0)

times_minutes = np.minimum((np.arange(n_bins) + 0.5) * bin_size, n_samples) / fs / 60
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for i, name in enumerate(FULL_RECORDING_CHANNELS):
    axes[i].fill_between(times_minutes, mins[i], maxs[i], alpha=0.8)
    axes[i].set_title(name)
    axes[i].set_ylabel("Raw units")
    axes[i].grid(alpha=0.3)
axes[-1].set_xlabel("Time (minutes)")
axes[-1].set_xlim(0, n_samples / fs / 60)
fig.suptitle("BCI2000 continuous signals, full recording (corrected decoding)")
plt.tight_layout()
plt.show()
print(f"Duration: {n_samples/fs/60:.3f} min | {n_samples:,} samples | {fs:g} Hz")